# 🥇 Gold Layer — Customer Orders
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/silver/`, applies business-logic transformations, and writes to `medallion/gold/`.

| Silver Table | Gold Table | Key Transformations |
|---|---|---|
| `silver_orders` | `gold_customer_orders` | Filter to order-level rows, map channels, extract discounts, derive subscription & sequence flags, cast numerics, rename columns |

## 0. Setup & Paths

In [22]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
Gold   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/gold


---
## 1. Customer Orders (`silver_orders` → `gold_customer_orders`)

### Step 1 — Load `silver_orders` & filter to order-level rows

In [23]:
# Load silver_orders 
df = pd.read_parquet(SILVER_DIR / 'silver_orders.parquet')

# Extract order-level rows, not SKU-level
orders = df[df['Top Row'].notna()].copy()

# Map Source to clean channel (To Be Confirmed)
channel_map = {
    '17473175553'                       : 'Lazada',
    '17346527233'                       : 'DTC',
    '2891397'                           : 'Marketplace',
    '5569197'                           : 'DTC',
    '4659545'                           : 'Email',
    '1814173'                           : 'Affiliate',
    '3890849'                           : 'Shop App',
    'web'                               : 'DTC',
    'shopee'                            : 'Shopee',
    'lazada'                            : 'Lazada',
    'pos'                               : 'POS',
    'tiktok'                            : 'TikTok',
    'subscription_contract'             : 'Subscription',
    'subscription_contract_checkout_one': 'Subscription',
    'shopify_draft_order'               : 'Draft Order',
    'Matrixify App'                     : 'Bulk Import',
}
orders['channel'] = orders['Source'].map(channel_map).fillna('Other')

# Load discount lookup from silver
order_discounts = pd.read_parquet(SILVER_DIR / 'silver_order_discounts_lookup.parquet')
orders = orders.merge(order_discounts, left_on='ID', right_on='order_id', how='left')

# Derive subscription flags from Tags
orders['is_subscription_order'] = orders['Tags'].str.contains('Subscription Order', na=False)
orders['is_recurring_order']    = orders['Tags'].str.contains('Recurring Order', na=False)

def extract_recurring_num(tag):
    if pd.isna(tag): return None
    match = re.search(r'Recurring Order #(\d+)', tag)
    return int(match.group(1)) if match else None

orders['recurring_order_num'] = orders['Tags'].apply(extract_recurring_num)

# Derive order sequence per customer
orders['Processed At'] = pd.to_datetime(orders['Processed At'], errors='coerce', utc=True)
orders = orders.sort_values(['Customer: ID', 'Processed At'])

orders['order_sequence']       = orders.groupby('Customer: ID').cumcount() + 1
orders['is_first_order']       = orders['order_sequence'] == 1
orders['prev_order_date']      = orders.groupby('Customer: ID')['Processed At'].shift(1)
orders['days_since_last_order']= (orders['Processed At'] - orders['prev_order_date']).dt.days

# Step 6: Cast numeric columns
for col in ['Price: Total', 'Price: Subtotal', 'Price: Total Discount',
            'Price: Total Shipping', 'Price: Total Refund']:
    orders[col] = pd.to_numeric(orders[col], errors='coerce').round(2)

# Select and rename final columns
gold_customer_orders = orders[[
    'ID', 'Name', 'Customer: ID', 'Processed At',
    'channel', 'Source',
    'Shipping: Country Code', 'Shipping: City',
    'Price: Total', 'Price: Subtotal', 'Price: Total Discount',
    'Price: Total Shipping', 'Price: Total Refund',
    'Payment: Status', 'Order Fulfillment Status',
    'Cancelled At', 'Cancel: Reason',
    'Browser: UTM Source', 'Browser: UTM Medium', 'Browser: UTM Campaign',
    'discount_code', 'discount_amount',
    'order_sequence', 'is_first_order',
    'days_since_last_order',
    'is_subscription_order', 'is_recurring_order', 'recurring_order_num',
    'Customer: Tags', 'Tags'
]].rename(columns={
    'ID'                      : 'order_id',
    'Name'                    : 'order_name',
    'Customer: ID'            : 'customer_id',
    'Customer: Tags'          : 'customer_tags',
    'Processed At'            : 'processed_at',
    'Shipping: Country Code'  : 'country_code',
    'Shipping: City'          : 'city',
    'Price: Total'            : 'price_total',
    'Price: Subtotal'         : 'price_subtotal',
    'Price: Total Discount'   : 'price_total_discount',
    'Price: Total Shipping'   : 'price_total_shipping',
    'Price: Total Refund'     : 'price_total_refund',
    'Payment: Status'         : 'payment_status',
    'Order Fulfillment Status': 'fulfillment_status',
    'Cancelled At'            : 'cancelled_at',
    'Cancel: Reason'          : 'cancel_reason',
    'Browser: UTM Source'     : 'utm_source',
    'Browser: UTM Medium'     : 'utm_medium',
    'Browser: UTM Campaign'   : 'utm_campaign',
    'Tags'                    : 'order_tags',
})

print(f"gold_customer_orders: {gold_customer_orders.shape[0]:,} rows × {gold_customer_orders.shape[1]} cols")
print(f"\nChannel breakdown:\n{gold_customer_orders['channel'].value_counts()}")
print(f"\nFirst orders: {gold_customer_orders['is_first_order'].sum():,}")
print(f"Repeat orders: {(~gold_customer_orders['is_first_order']).sum():,}")
print(f"Orders with discount: {gold_customer_orders['discount_code'].notna().sum():,}")

gold_customer_orders.to_parquet(GOLD_DIR / 'gold_customer_orders.parquet', index=False)
print("\n✅ Saved: gold_customer_orders.parquet")

gold_customer_orders: 28,054 rows × 30 cols

Channel breakdown:
channel
DTC             11059
Lazada           7339
Other            2976
Subscription     1905
Marketplace      1590
Draft Order      1227
Shopee            946
Email             476
POS               464
TikTok             34
Affiliate          19
Shop App           19
Name: count, dtype: int64

First orders: 13,885
Repeat orders: 14,169
Orders with discount: 8,497

✅ Saved: gold_customer_orders.parquet


In [24]:
display(gold_customer_orders.head(10))

,order_id,order_name,customer_id,processed_at,channel,Source,country_code,city,price_total,price_subtotal,price_total_discount,price_total_shipping,price_total_refund,payment_status,fulfillment_status,cancelled_at,cancel_reason,utm_source,utm_medium,utm_campaign,discount_code,discount_amount,order_sequence,is_first_order,days_since_last_order,is_subscription_order,is_recurring_order,recurring_order_num,customer_tags,order_tags
13507,5291518329087,LPSG-21248,6325455978751,2023-08-01 10:54:08+00:00,Draft Order,shopify_draft_order,sg,Singapore,11.96,11.96,67.84,0.0,11.96,refunded,None,2023-08-01 10:55:24+00:00,customer,None,None,None,NaN,NaN,1.0,True,NaN,False,False,NaN,"Has active product subscriptions, male, newsletter, RFM Group _no_rfm, RFM S...",None
13508,5291520196863,LPSG-21249,6325455978751,2023-08-01 10:58:02+00:00,DTC,web,sg,Singapore,33.92,33.92,0.00,0.0,33.92,refunded,None,2023-08-01 11:04:05+00:00,customer,None,None,None,NaN,NaN,2.0,False,0.0,False,True,1.0,"Has active product subscriptions, male, newsletter, RFM Group _no_rfm, RFM S...","Recurring Order #1, Yotpo Subscriptions"
13602,5322207461631,LPSG-21343,6325455978751,2023-08-29 07:15:15+00:00,Subscription,subscription_contract,sg,Singapore,33.92,33.92,0.00,0.0,33.92,refunded,None,2023-08-29 07:32:12+00:00,customer,None,None,None,NaN,NaN,3.0,False,27.0,True,True,2.0,"Has active product subscriptions, male, newsletter, RFM Group _no_rfm, RFM S...","Recurring Order #2, Subscription Order, Yotpo Subscriptions"
9619,4992506265855,LPSG-6339,6327387259135,2022-06-29 12:34:53+00:00,DTC,17346527233,sg,Singapore,1693.12,1693.12,0.00,0.0,NaN,paid,fulfilled,NaT,None,None,None,None,NaN,NaN,1.0,True,NaN,False,False,NaN,"female, RFM Group Don Juan, RFM Score 115",None
3745,4992632127743,LPSG-8071,6327388733695,2021-10-03 17:43:25+00:00,DTC,17346527233,sg,Singapore,222.20,222.20,0.00,0.0,NaN,paid,fulfilled,NaT,None,None,None,None,NaN,NaN,1.0,True,NaN,False,False,NaN,"loyal-app_about_to_sleep, loyal-app_new_in_segment, male, Purchased:prime wh...",None
15638,5618712019199,LPSG-22204,6327388733695,2024-05-03 14:43:37+00:00,DTC,web,sg,Singapore,113.81,113.81,5.99,0.0,NaN,paid,fulfilled,NaT,None,[all] bdsequence,email,email _hash_7 (lnxerc),Buy 2 Save 5%,5.99,2.0,False,942.0,False,False,NaN,"loyal-app_about_to_sleep, loyal-app_new_in_segment, male, Purchased:prime wh...",None
5000,4992730300671,LPSG-9326,6327388799231,2021-02-04 17:18:42+00:00,DTC,17346527233,sg,Singapore,19.00,19.00,0.00,0.0,NaN,paid,fulfilled,NaT,None,None,None,None,NaN,NaN,1.0,True,NaN,False,False,NaN,"female, RFM Group Ex lover, RFM Score 133, STARTER",None
3668,4992626262271,LPSG-7994,6327388799231,2021-10-10 18:39:28+00:00,DTC,17346527233,sg,Singapore,21.75,21.75,0.00,0.0,NaN,paid,fulfilled,NaT,None,None,None,None,NaN,NaN,2.0,False,248.0,False,False,NaN,"female, RFM Group Ex lover, RFM Score 133, STARTER",None
3490,4992612761855,LPSG-7816,6327388799231,2021-11-04 20:27:46+00:00,DTC,17346527233,sg,Singapore,52.32,47.42,0.00,4.9,NaN,paid,fulfilled,NaT,None,None,None,None,NaN,NaN,3.0,False,25.0,False,False,NaN,"female, RFM Group Ex lover, RFM Score 133, STARTER",None
4883,4992722010367,LPSG-9208,6327389520127,2021-03-03 21:16:41+00:00,DTC,17346527233,sg,Singapore,29.00,29.00,0.00,0.0,NaN,paid,fulfilled,NaT,None,None,None,None,NaN,NaN,1.0,True,NaN,False,False,NaN,"male, RFM Group Breakup, RFM Score 111, STARTER",None
